# Linear Regression — Quality & Commercial Score

Trains a Ridge regression model for each target using a `ColumnTransformer` that scales
only the continuous features. Saves the fitted pipelines and a markdown report to `models/linear_regression/`.

In [1]:
# ── Constants ──────────────────────────────────────────────────────────────
import os

TRAIN_RATIO = float(os.environ.get("LUDOMETRICS_TRAIN_RATIO", "0.80"))
TEST_RATIO = round(1.0 - TRAIN_RATIO, 10)
RANDOM_STATE = 42
TOP_N_FEATURES = 10

# Derived split label used in output filenames, e.g. "80_20"
SPLIT_LABEL = f"{int(TRAIN_RATIO * 100)}_{int(TEST_RATIO * 100)}"
SPLIT_DISPLAY = f"{int(TRAIN_RATIO * 100)}/{int(TEST_RATIO * 100)}"

In [2]:
import sys
import time

sys.path.insert(0, "..")

import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

from utils.train_utils import update_results_table

## 1. Load data

In [3]:
df = pd.read_csv("../data/games_processed.csv")
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")

Loaded 21,925 rows × 403 columns


## 2. Define feature columns

In [4]:
CONTINUOUS_COLS = [
    "GameWeight",
    "MinPlayers",
    "MaxPlayers",
    "ComAgeRec",
    "MfgAgeRec",
    "MfgPlaytime",
    "ComMinPlaytime",
    "ComMaxPlaytime",
]

DROP_COLS = ["BGGId", "quality_score", "commercial_score"]
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS]
TARGETS = ["quality_score", "commercial_score"]

X = df[FEATURE_COLS]
print(
    f"Features: {len(FEATURE_COLS)} columns ({len(CONTINUOUS_COLS)} continuous, {len(FEATURE_COLS) - len(CONTINUOUS_COLS)} binary)"
)

Features: 400 columns (8 continuous, 392 binary)


## 3. Train / test split

In [5]:
X_train, X_test = train_test_split(X, test_size=TEST_RATIO, random_state=RANDOM_STATE)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

Train: 17,540  |  Test: 4,385


## 4. Train one model per target

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def build_linear_pipeline(continuous_cols: list[str], all_feature_cols: list[str]) -> Pipeline:
    """StandardScaler on continuous cols, passthrough on binary cols."""
    preprocessor = ColumnTransformer(
        transformers=[
            ("scale", StandardScaler(), continuous_cols),
        ],
        remainder="passthrough",
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

    return Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", RidgeCV()),
        ]
    )

In [7]:
results = {}

for target in TARGETS:
    y_train = df.loc[X_train.index, target]
    y_test = df.loc[X_test.index, target]

    pipeline = build_linear_pipeline(CONTINUOUS_COLS, FEATURE_COLS)
    t0 = time.monotonic()
    pipeline.fit(X_train, y_train)
    elapsed = time.monotonic() - t0

    y_pred = pipeline.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    results[target] = {"pipeline": pipeline, "rmse": rmse, "r2": r2, "elapsed": elapsed, "y_pred": y_pred, "y_test": y_test}
    print(f"{target:20s}  RMSE={rmse:.4f}  R²={r2:.4f}")

quality_score         RMSE=2.7841  R²=0.5124


commercial_score      RMSE=10.3130  R²=0.4964


## 5. Save models

In [8]:
MODEL_DIR = Path("../results/models/linear_regression")

for target in TARGETS:
    out_path = MODEL_DIR / f"{target}_{SPLIT_LABEL}.pkl"
    joblib.dump(results[target]["pipeline"], out_path)
    print(f"Saved {out_path}")

    predictions = pd.DataFrame({
        "BGGId": df.loc[X_test.index, "BGGId"].values,
        "actual": results[target]["y_test"].values,
        "predicted": results[target]["y_pred"],
    })
    pred_path = MODEL_DIR / f"{target}_{SPLIT_LABEL}_predictions.csv"
    predictions.to_csv(pred_path, index=False)
    print(f"Saved {pred_path}")

Saved ../results/models/linear_regression/quality_score_80_20.pkl
Saved ../results/models/linear_regression/quality_score_80_20_predictions.csv
Saved ../results/models/linear_regression/commercial_score_80_20.pkl
Saved ../results/models/linear_regression/commercial_score_80_20_predictions.csv


## 6. Write report

In [9]:
# Top-10 coefficients by absolute magnitude (averaged across both targets)
coef_frames = []
for target in TARGETS:
    pipeline = results[target]["pipeline"]
    model = pipeline.named_steps["model"]
    feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
    coef_frames.append(pd.Series(np.abs(model.coef_), index=feature_names, name=target))

avg_abs_coef = pd.concat(coef_frames, axis=1).mean(axis=1).sort_values(ascending=False)
top_features = list(avg_abs_coef.head(TOP_N_FEATURES).items())

metrics = {
    target: {"rmse": results[target]["rmse"], "r2": results[target]["r2"]}
    for target in TARGETS
}

print()

RESULTS_DIR = Path("../results")

for target in TARGETS:
    elapsed = results[target]["elapsed"]
    minutes, seconds = divmod(elapsed, 60)
    time_label = (
        f"{int(minutes)}m {seconds:.1f}s" if minutes >= 1 else f"{seconds:.1f}s"
    )
    update_results_table(
        RESULTS_DIR / f"{target}.md",
        algorithm="Linear Regression (RidgeCV)",
        split=SPLIT_DISPLAY,
        train_size=len(X_train),
        test_size=len(X_test),
        training_time=time_label,
        rmse=results[target]["rmse"],
        r2=results[target]["r2"],
    )
    print(f"Updated results/{target}.md")

Updated results/quality_score.md


Updated results/commercial_score.md
